In [0]:
from pyspark.sql.functions import *

# ==========================================================
# Read Silver Tables
# ==========================================================

inpatient = spark.read.table(
    "healthcare_claims_catalog.silver.inpatient"
)

outpatient = spark.read.table(
    "healthcare_claims_catalog.silver.outpatient"
)

carrier = spark.read.table(
    "healthcare_claims_catalog.silver.carrier"
)

pde = spark.read.table(
    "healthcare_claims_catalog.silver.pde"
)

# ==========================================================
# Read Gold Dimensions
# ==========================================================

dim_beneficiary = spark.read.table(
    "healthcare_claims_catalog.gold.dim_beneficiary"
)

dim_provider = spark.read.table(
    "healthcare_claims_catalog.gold.dim_provider"
)

dim_date = spark.read.table(
    "healthcare_claims_catalog.gold.dim_date"
)

dim_diagnosis = spark.read.table(
    "healthcare_claims_catalog.gold.dim_diagnosis"
)

# ==========================================================
# Standardize Inpatient
# FIX: CLM_PMT_AMT is what Medicare actually paid — there is no
# separate "billed amount" field in this dataset. CLAIM_AMOUNT is
# therefore the paid amount PLUS the patient's deductible/coinsurance
# (i.e. the full value of the claim). INSURANCE_PAYMENT is CLM_PMT_AMT
# directly, not NCH_PRMRY_PYR_CLM_PD_AMT (that field is for the rare
# case of a DIFFERENT primary payer when Medicare is secondary — it's
# ~always $0 and was being wrongly used as "insurance payment").
# ==========================================================

inpatient_fact = inpatient.select(

    col("CLM_ID"),

    col("DESYNPUF_ID"),

    col("PRVDR_NUM").alias("PROVIDER_ID"),

    col("CLM_FROM_DT").alias("CLAIM_DATE"),

    col("ICD9_DGNS_CD_1").alias("DIAGNOSIS_CODE"),

    (
        coalesce(col("CLM_PMT_AMT").cast("double"), lit(0.0))
        + coalesce(col("NCH_BENE_IP_DDCTBL_AMT").cast("double"), lit(0.0))
        + coalesce(col("NCH_BENE_PTA_COINSRNC_LBLTY_AM").cast("double"), lit(0.0))
    ).alias("CLAIM_AMOUNT"),

    col("CLM_PMT_AMT").cast("double").alias("INSURANCE_PAYMENT"),

    lit("INPATIENT").alias("CLAIM_TYPE")

)

# ==========================================================
# Standardize Outpatient
# Same fix as Inpatient, using Part B deductible/coinsurance fields
# ==========================================================

outpatient_fact = outpatient.select(

    col("CLM_ID"),

    col("DESYNPUF_ID"),

    col("PRVDR_NUM").alias("PROVIDER_ID"),

    col("CLM_FROM_DT").alias("CLAIM_DATE"),

    col("ICD9_DGNS_CD_1").alias("DIAGNOSIS_CODE"),

    (
        coalesce(col("CLM_PMT_AMT").cast("double"), lit(0.0))
        + coalesce(col("NCH_BENE_PTB_DDCTBL_AMT").cast("double"), lit(0.0))
        + coalesce(col("NCH_BENE_PTB_COINSRNC_AMT").cast("double"), lit(0.0))
    ).alias("CLAIM_AMOUNT"),

    col("CLM_PMT_AMT").cast("double").alias("INSURANCE_PAYMENT"),

    lit("OUTPATIENT").alias("CLAIM_TYPE")

)

# ==========================================================
# Standardize Carrier
# FIX: CLAIM_AMOUNT now uses the allowed charge (the actual total
# claim value), not a duplicate of the payment amount. INSURANCE_PAYMENT
# stays as the line payment. This makes PATIENT_PAYMENT (computed later
# as CLAIM_AMOUNT - INSURANCE_PAYMENT) reflect the real deductible +
# coinsurance instead of always being $0.
# NOTE: still line-1-only (out of up to 13 line items per claim) — a
# known simplification, not something this fix addresses.
# NOTE: PROVIDER_ID uses TAX_NUM_1 here, a different ID scheme than
# Inpatient/Outpatient's PRVDR_NUM — confirm dim_provider actually has
# TAX_NUM-keyed rows, or Carrier claims will get a NULL PROVIDER_KEY.
# ==========================================================

carrier_fact = carrier.select(

    col("CLM_ID"),

    col("DESYNPUF_ID"),

    col("TAX_NUM_1").alias("PROVIDER_ID"),

    col("CLM_FROM_DT").alias("CLAIM_DATE"),

    col("ICD9_DGNS_CD_1").alias("DIAGNOSIS_CODE"),

    col("LINE_ALOWD_CHRG_AMT_1")
        .cast("double")
        .alias("CLAIM_AMOUNT"),

    col("LINE_NCH_PMT_AMT_1")
        .cast("double")
        .alias("INSURANCE_PAYMENT"),

    lit("CARRIER").alias("CLAIM_TYPE")

)

# ==========================================================
# Standardize PDE
# Unchanged — this one was already correct
# ==========================================================

pde_fact = pde.select(

    col("PDE_ID").alias("CLM_ID"),

    col("DESYNPUF_ID"),

    lit(None).cast("string").alias("PROVIDER_ID"),

    col("SRVC_DT").alias("CLAIM_DATE"),

    lit(None).cast("string").alias("DIAGNOSIS_CODE"),

    col("TOT_RX_CST_AMT")
        .cast("double")
        .alias("CLAIM_AMOUNT"),

    (
        col("TOT_RX_CST_AMT")-
        col("PTNT_PAY_AMT")
    ).cast("double").alias("INSURANCE_PAYMENT"),

    lit("PDE").alias("CLAIM_TYPE")

)

# ==========================================================
# Merge All Claims
# ==========================================================

fact_claims = (
    inpatient_fact
    .unionByName(outpatient_fact)
    .unionByName(carrier_fact)
    .unionByName(pde_fact)
)

# ==========================================================
# Patient Responsibility
# ==========================================================

fact_claims = fact_claims.withColumn(
    "PATIENT_PAYMENT",
    greatest(
        lit(0.0),
        coalesce(col("CLAIM_AMOUNT"), lit(0.0))
        -
        coalesce(col("INSURANCE_PAYMENT"), lit(0.0))
    )
)

# ==========================================================
# Join Beneficiary Dimension
# (safe now — dim_beneficiary has one row per DESYNPUF_ID after the
# dim_beneficiary fix)
# ==========================================================

fact_claims = fact_claims.join(

    dim_beneficiary.select(

        "DESYNPUF_ID",

        "BENEFICIARY_KEY"

    ),

    "DESYNPUF_ID",

    "left"

)

# ==========================================================
# Join Provider Dimension
# ==========================================================

fact_claims = fact_claims.join(

    dim_provider.select(

        "PROVIDER_ID",

        "PROVIDER_KEY"

    ),

    "PROVIDER_ID",

    "left"

)

# ==========================================================
# Join Date Dimension
# ==========================================================

fact_claims = fact_claims.withColumn(

    "DATE_KEY",

    date_format(

        "CLAIM_DATE",

        "yyyyMMdd"

    ).cast("int")

)

fact_claims = fact_claims.join(

    dim_date.select(

        "DATE_KEY"

    ),

    "DATE_KEY",

    "left"

)

# ==========================================================
# Join Diagnosis Dimension
# ==========================================================

fact_claims = fact_claims.join(

    dim_diagnosis.select(

        "DIAGNOSIS_CODE",

        "DIAGNOSIS_KEY"

    ),

    "DIAGNOSIS_CODE",

    "left"

)

# ==========================================================
# Gold Audit Columns
# ==========================================================

fact_claims = (

    fact_claims

    .withColumn(

        "GOLD_CREATED_TIMESTAMP",

        current_timestamp()

    )

)

# ==========================================================
# Claim Status
# ==========================================================

fact_claims = fact_claims.withColumn(
    "CLAIM_STATUS",
    when(col("CLAIM_AMOUNT").isNull(), "PENDING REVIEW")

    .when(col("CLAIM_AMOUNT") <= 0, "DENIED")

    .when(col("INSURANCE_PAYMENT").isNull(), "PENDING REVIEW")

    .when(col("INSURANCE_PAYMENT") == 0, "DENIED")

    .otherwise("APPROVED")
)

# ==========================================================
# Payment Status
# ==========================================================

fact_claims = fact_claims.withColumn(
    "PAYMENT_STATUS",
    when(col("INSURANCE_PAYMENT").isNull(), "Unknown")

    .when(col("INSURANCE_PAYMENT") == 0, "Unpaid")

    .when(col("INSURANCE_PAYMENT") >= col("CLAIM_AMOUNT"), "Fully Paid")

    .otherwise("Partially Paid")
)

# ==========================================================
# Approval Percentage
# ==========================================================

fact_claims = fact_claims.withColumn(
    "APPROVAL_PERCENT",
    when(
        col("CLAIM_AMOUNT") > 0,
        round(
            (col("INSURANCE_PAYMENT") / col("CLAIM_AMOUNT")) * 100,
            2
        )
    ).otherwise(lit(0))
)

# ==========================================================
# Denial Reason
# ==========================================================

fact_claims = fact_claims.withColumn(
    "DENIAL_REASON",

    when(
        col("CLAIM_STATUS") == "DENIED",

        when(
            pmod(abs(hash(col("CLM_ID"))), 10).isin(0, 1, 2),
            "Invalid Claim Amount"
        )
        .when(
            pmod(abs(hash(col("CLM_ID"))), 10).isin(3, 4),
            "No Insurance Payment"
        )
        .when(
            pmod(abs(hash(col("CLM_ID"))), 10) == 5,
            "Missing Documentation"
        )
        .when(
            pmod(abs(hash(col("CLM_ID"))), 10) == 6,
            "Prior Authorization Required"
        )
        .when(
            pmod(abs(hash(col("CLM_ID"))), 10) == 7,
            "Service Not Covered"
        )
        .otherwise(
            "Eligibility Issue"
        )

    ).otherwise(
        lit(None).cast("string")
    )
)

# ==========================================================
# Risk Level
# NOTE (not fixed here, flagged for your team's decision): these
# absolute thresholds mix claim types with very different dollar
# scales (PDE averages ~$60, Carrier is line-1-only, Inpatient
# averages ~$9,500) — nearly everything will land in "Low" as a
# result. Consider per-claim-type or percentile-based thresholds
# if you want this metric to be more informative.
# ==========================================================

fact_claims = fact_claims.withColumn(
    "RISK_LEVEL",

    when(col("CLAIM_AMOUNT") >= 50000, "High")

    .when(col("CLAIM_AMOUNT") >= 10000, "Medium")

    .otherwise("Low")
)

# ==========================================================
# Reorder Columns
# ==========================================================

fact_claims = fact_claims.select(

    "CLM_ID",
    "BENEFICIARY_KEY",
    "PROVIDER_KEY",
    "DATE_KEY",
    "DIAGNOSIS_KEY",

    "CLAIM_TYPE",

    "CLAIM_STATUS",
    "PAYMENT_STATUS",
    "DENIAL_REASON",
    "RISK_LEVEL",

    "CLAIM_AMOUNT",
    "INSURANCE_PAYMENT",
    "PATIENT_PAYMENT",
    "APPROVAL_PERCENT",

    "GOLD_CREATED_TIMESTAMP"
)

# ==========================================================
# Write Gold Table
# ==========================================================

fact_claims.write \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.format("delta") \
.saveAsTable(
"healthcare_claims_catalog.gold.fact_claims"
)

# ==========================================================
# Validation
# ==========================================================

print("="*60)

print("FACT CLAIMS CREATED SUCCESSFULLY")

print("="*60)

total = fact_claims.count()
print("Total Records :", total)

print("\nClaim Status breakdown:")
fact_claims.groupBy("CLAIM_STATUS").count().show(truncate=False)

print("\nInsurance vs Patient payment totals:")
fact_claims.select(
    sum("INSURANCE_PAYMENT").alias("total_insurance"),
    sum("PATIENT_PAYMENT").alias("total_patient")
).show(truncate=False)

fact_claims.printSchema()

fact_claims.show(10,False)

fact_claims.groupBy(
    "CLAIM_STATUS",
    "DENIAL_REASON"
).count().orderBy(
    "CLAIM_STATUS",
    "DENIAL_REASON"
).show(truncate=False)

FACT CLAIMS CREATED SUCCESSFULLY
Total Records : 11073217

Claim Status breakdown:
+------------+-------+
|CLAIM_STATUS|count  |
+------------+-------+
|DENIED      |2204457|
|APPROVED    |8868760|
+------------+-------+


Insurance vs Patient payment totals:
+---------------+-------------+
|total_insurance|total_patient|
+---------------+-------------+
|1.38394106E9   |3.14783402E8 |
+---------------+-------------+

root
 |-- CLM_ID: string (nullable = true)
 |-- BENEFICIARY_KEY: integer (nullable = true)
 |-- PROVIDER_KEY: integer (nullable = true)
 |-- DATE_KEY: integer (nullable = true)
 |-- DIAGNOSIS_KEY: integer (nullable = true)
 |-- CLAIM_TYPE: string (nullable = false)
 |-- CLAIM_STATUS: string (nullable = false)
 |-- PAYMENT_STATUS: string (nullable = false)
 |-- DENIAL_REASON: string (nullable = true)
 |-- RISK_LEVEL: string (nullable = false)
 |-- CLAIM_AMOUNT: double (nullable = true)
 |-- INSURANCE_PAYMENT: double (nullable = true)
 |-- PATIENT_PAYMENT: double (nullable =